In [1]:
import os
import json
from collections import Counter
from pathlib import Path
import numpy as np
import torch
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from vul_detector import VulDetector
from vul_trainer import VulTrainerManual
from imblearn.over_sampling import SMOTE


/Users/sumaila/uakron/research/code/vuldetector/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sumaila/uakron/research/code/vuldetector/env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# -------------------------
# 0) Load JSONL -> HF Dataset
# -------------------------
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

# Set this to your folder containing train/val/test jsonl from preprocessing
DATASET_DIR = os.path.join(os.getcwd(), "data/processed_data")
assert os.path.isdir(DATASET_DIR), f"Dataset dir not found: {DATASET_DIR}"

train_dataset = Dataset.from_list(load_jsonl(os.path.join(DATASET_DIR, "train.jsonl")))
valid_dataset = Dataset.from_list(load_jsonl(os.path.join(DATASET_DIR, "val.jsonl")))
test_dataset = Dataset.from_list(load_jsonl(os.path.join(DATASET_DIR, "test.jsonl")))

print("Loaded:", len(train_dataset), len(valid_dataset), len(test_dataset))
print("Example row keys:", train_dataset.column_names)
print("Example:", {k: train_dataset[0][k] for k in ["id", "project", "target", "answer_text"]})

Loaded: 110878 87122 26300
Example row keys: ['id', 'project', 'target', 'func_clean', 'prompt', 'answer_text']
Example: {'id': '7053ed116a987e80', 'project': 'openssl', 'target': 1, 'answer_text': 'vulnerable'}


In [5]:
sum(1 for x in train_dataset['target'] if x == 1)

3214

In [6]:

sum(1 for x in train_dataset['target'] if x == 0)

107664

In [ ]:
# -------------------------
# 2) Tokenizer + Map
# -------------------------
model_name = "microsoft/unixcoder-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

MAX_LEN = 1024  # start 256; try 384 if needed (1024 will likely OOM)

# def tokenize_batch(batch):
#     texts = [c.strip() for c in batch["func_clean"]]

#     enc = tokenizer(
#         texts,
#         truncation=True,
#         max_length=MAX_LEN,
#         padding="max_length",
#         add_special_tokens=True,
#     )

#     # Labels: prefer numeric target
#     if "target" in batch:
#         enc["labels"] = [int(x) for x in batch["target"]]
#     else:
#         enc["labels"] = [
#             1 if a.strip().lower() == "vulnerable" else 0
#             for a in batch["answer_text"]
#         ]
#     return enc
def tokenize_batch(batch):
    texts = [c.strip() for c in batch["func_clean"]]
    enc = tokenizer(texts, truncation=True, max_length=MAX_LEN, padding="max_length")

    # debug: detect truncation
    # if tokenizer returns overflowing info, use return_overflowing_tokens=True
    # easiest: compare pre-length vs MAX_LEN
    raw_lens = [len(tokenizer(t).input_ids) for t in texts]
    trunc_rate = sum(l > MAX_LEN for l in raw_lens) / len(raw_lens)
    print("trunc_rate:", trunc_rate)

    enc["labels"] = [int(x) for x in batch["target"]]
    return enc
train_tok = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names,
)
valid_tok = valid_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=valid_dataset.column_names,
)
test_tok = test_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=test_dataset.column_names,
)

In [ ]:

# compute embeddings for train set, apply SMOTE to embeddings, produce a balanced DataLoader

def balance_embeddings_with_smote(batcher, emb_model, embed_device):
    emb_model.to(embed_device)
    emb_model.eval()
    embs = []
    ys = []
    with torch.no_grad():
        for b in batcher:
            input_ids = b["input_ids"].to(embed_device)
            attention_mask = b["attention_mask"].to(embed_device)
            out = emb_model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            # use pooler_output if available else mean-pool last hidden state
            if hasattr(out, "pooler_output") and out.pooler_output is not None:
                pooled = out.pooler_output
            else:
                pooled = out.last_hidden_state.mean(dim=1)
            embs.append(pooled.cpu())
            ys.append(b["labels"].cpu())

    X = torch.cat(embs).numpy()
    y = torch.cat(ys).numpy()

    print("Before SMOTE:", Counter(y))

    smote = SMOTE(sampling_strategy="auto", k_neighbors=3, random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)

    print("After SMOTE:", Counter(y_smote))

    # create a DataLoader of embeddings+labels (for training a classifier on embeddings)
    X_smote_t = torch.tensor(X_smote, dtype=torch.float32)
    y_smote_t = torch.tensor(y_smote, dtype=torch.long)
    smote_dataset = torch.utils.data.TensorDataset(X_smote_t, y_smote_t)
    smote_loader = DataLoader(smote_dataset, batch_size=32, shuffle=True)
    return smote_loader


In [ ]:
# -------------------------
# 3) DataLoaders + class weights
# -------------------------
embed_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Embedding device:", embed_device)
emb_model = AutoModel.from_pretrained(model_name)

columns = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=columns)
valid_tok.set_format(type="torch", columns=columns)
test_tok.set_format(type="torch", columns=columns)

train_loader = DataLoader(train_tok, batch_size=16, shuffle=True)
val_loader = DataLoader(valid_tok, batch_size=32)
test_loader = DataLoader(test_tok, batch_size=32)

balance_train_loader = balance_embeddings_with_smote(train_loader, emb_model, embed_device)
balance_val_loader = balance_embeddings_with_smote(val_loader, emb_model, embed_device)
balance_test_loader = balance_embeddings_with_smote(test_loader, emb_model, embed_device)

label_counts = Counter(train_dataset["target"])
total = sum(label_counts.values())
class_weights = [total / (2 * label_counts[i]) for i in range(2)]
print("Class weights:", class_weights)
print("Batches -> train:", len(balance_train_loader), "val:", len(balance_val_loader), "test:", len(balance_test_loader))

Class weights: [0.5139670063304544, 18.39932603201348]
Batches -> train: 5460 val: 2045 test: 714


In [ ]:
# -------------------------
# 4) Train vulnerability detector
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = VulDetector(model_name=model_name, num_labels=2)
trainer = VulTrainerManual(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    class_weights=class_weights,
    learning_rate=1e-5,
    num_epochs=12,
    loss_type="focal",
    focal_gamma=1.5,
)

#trainer.train()

Using device: cpu


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


1000000000000000019884624838656


In [ ]:
# -------------------------
# 5) Evaluate on validation/test splits
# -------------------------
def evaluate_loader(loader):
    model.eval()
    total_loss = 0.0
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels_batch = batch.pop("labels")
            outputs = model(**batch)
            logits = outputs.logits
            loss = trainer.criterion(logits, labels_batch)
            total_loss += loss.item()
            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            labels.extend(labels_batch.cpu().tolist())
    metrics = trainer.compute_metrics(np.array(preds), np.array(labels))
    metrics["loss"] = total_loss / max(len(loader), 1)
    return metrics

best_ckpts = sorted(Path(".").glob("best_model_epoch_*.pt"), key=lambda p: p.stat().st_mtime)
if best_ckpts:
    best_ckpt = best_ckpts[-1]
    model.load_state_dict(torch.load(best_ckpt, map_location=device))
    model.to(device)
    print(f"Loaded best checkpoint: {best_ckpt}")
else:
    print("No saved checkpoints found; evaluating current model state.")

val_metrics = evaluate_loader(val_loader)
print("Validation metrics:", val_metrics)

test_metrics = evaluate_loader(test_loader)
print("Test metrics:", test_metrics)